<a href="https://colab.research.google.com/github/muhsinasafeeth/Exchange-Rate-Prediction-INR---AED-/blob/main/INR_AED_Exchange_Rate_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💱 INR to AED Exchange Rate Prediction

![Python](https://img.shields.io/badge/Python-3.8%2B-blue) ![ML](https://img.shields.io/badge/ML-Time%20Series-orange) ![Deep Learning](https://img.shields.io/badge/Deep%20Learning-LSTM-red)

## Objective
Predict future **INR → AED (Indian Rupee to UAE Dirham)** exchange rates using four machine learning and deep learning approaches:

| # | Algorithm | Type |
|---|-----------|------|
| 1 | **Linear Regression** | Classical ML |
| 2 | **Random Forest** | Ensemble ML |
| 3 | **ARIMA** | Statistical Time Series |
| 4 | **LSTM** | Deep Learning |

---

## Table of Contents
1. [Setup & Data Collection](#1-setup--data-collection)
2. [Exploratory Data Analysis (EDA)](#2-exploratory-data-analysis)
3. [Preprocessing & Feature Engineering](#3-preprocessing--feature-engineering)
4. [Linear Regression](#4-linear-regression)
5. [Random Forest Regressor](#5-random-forest-regressor)
6. [ARIMA](#6-arima-time-series)
7. [LSTM (Deep Learning)](#7-lstm-deep-learning)
8. [Model Comparison & Future Prediction](#8-model-comparison--future-prediction)

---
## 1. Setup & Data Collection

### 1.1 Install & Import Libraries

We use `yfinance` to fetch **real historical INR/AED exchange rate data** directly from Yahoo Finance. This gives us daily exchange rate data going back several years.

| Library | Purpose |
|---------|--------|
| `yfinance` | Fetch real historical forex data |
| `pandas` / `numpy` | Data manipulation |
| `matplotlib` / `seaborn` | Visualization |
| `sklearn` | Linear Regression, Random Forest, metrics |
| `statsmodels` | ARIMA time series model |
| `tensorflow/keras` | LSTM deep learning model |

In [1]:
# ─── Install required libraries (run once) ───────────────────────────────────
!pip install yfinance statsmodels tensorflow scikit-learn matplotlib seaborn --quiet
print("✅ All libraries installed!")

✅ All libraries installed!


In [2]:
# ─── Import all libraries ────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Data collection
import yfinance as yf

# ML Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# ARIMA
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ All libraries imported successfully!")
print(f"   TensorFlow version : {tf.__version__}")

✅ All libraries imported successfully!
   TensorFlow version : 2.19.0


### 1.2 Fetch Real INR/AED Data

Yahoo Finance uses the ticker **`INRAED=X`** for the INR to AED forex pair.
We fetch **5 years** of daily closing price data.

In [ ]:
# ─── Fetch INR to AED Historical Data ────────────────────────────────────────
# Ticker: INRAED=X  → INR to AED forex pair on Yahoo Finance
# Period: 5 years of daily data

ticker = yf.Ticker("INRAED=X")
df_raw = ticker.history(period="5y", interval="1d")

# Keep only the closing price
df = df_raw[['Close']].copy()
df.index = pd.to_datetime(df.index).tz_localize(None)  # Remove timezone
df.columns = ['INR_AED']
df.dropna(inplace=True)

print(f"📊 Data fetched successfully!")
print(f"   Total records  : {len(df)}")
print(f"   Date range     : {df.index[0].date()} → {df.index[-1].date()}")
print(f"   Current rate   : 1 INR = {df['INR_AED'].iloc[-1]:.5f} AED")
print(f"\n📋 First 5 rows:")
df.head()

---
## 2. Exploratory Data Analysis

Before building any model, we explore the data to understand:
- Overall trend of INR/AED over time
- Volatility and distribution of rates
- Rolling averages (moving averages)
- Whether the series is stationary (required for ARIMA)

In [ ]:
# ─── Basic Statistics ────────────────────────────────────────────────────────
print("📌 Descriptive Statistics:")
print(df.describe().round(6))

In [ ]:
# ─── Historical Exchange Rate Plot ───────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Full historical rate
axes[0].plot(df.index, df['INR_AED'], color='#2980b9', linewidth=1.2, label='INR/AED Rate')
axes[0].set_title('INR to AED Exchange Rate — 5 Year History', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Rate (1 INR = ? AED)')
axes[0].legend()

# Plot 2: With Moving Averages
df['MA_30']  = df['INR_AED'].rolling(window=30).mean()   # 30-day MA
df['MA_90']  = df['INR_AED'].rolling(window=90).mean()   # 90-day MA
df['MA_200'] = df['INR_AED'].rolling(window=200).mean()  # 200-day MA

axes[1].plot(df.index, df['INR_AED'], color='#bdc3c7', linewidth=0.8, alpha=0.7, label='Daily Rate')
axes[1].plot(df.index, df['MA_30'],  color='#e74c3c', linewidth=1.5, label='30-day MA')
axes[1].plot(df.index, df['MA_90'],  color='#f39c12', linewidth=1.5, label='90-day MA')
axes[1].plot(df.index, df['MA_200'], color='#2ecc71', linewidth=1.5, label='200-day MA')
axes[1].set_title('Exchange Rate with Moving Averages', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Rate')
axes[1].legend()

# Plot 3: Daily Returns (% change)
df['Daily_Return'] = df['INR_AED'].pct_change() * 100
axes[2].plot(df.index, df['Daily_Return'], color='#9b59b6', linewidth=0.8)
axes[2].axhline(0, color='black', linewidth=1)
axes[2].fill_between(df.index, df['Daily_Return'], 0,
                     where=df['Daily_Return'] > 0, color='#2ecc71', alpha=0.4, label='Positive')
axes[2].fill_between(df.index, df['Daily_Return'], 0,
                     where=df['Daily_Return'] < 0, color='#e74c3c', alpha=0.4, label='Negative')
axes[2].set_title('Daily Returns (%)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Return (%)')
axes[2].legend()

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Distribution & Volatility Analysis ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution of rates
axes[0].hist(df['INR_AED'], bins=50, color='#3498db', edgecolor='white')
axes[0].set_title('Distribution of INR/AED Rate', fontweight='bold')
axes[0].set_xlabel('Rate')
axes[0].set_ylabel('Frequency')

# Rolling Volatility (30-day std)
df['Volatility'] = df['INR_AED'].rolling(window=30).std()
axes[1].plot(df.index, df['Volatility'], color='#e67e22', linewidth=1.2)
axes[1].set_title('30-Day Rolling Volatility', fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Std Dev')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)

# Yearly boxplot
df['Year'] = df.index.year
yearly_data = [df[df['Year'] == y]['INR_AED'].values for y in sorted(df['Year'].unique())]
axes[2].boxplot(yearly_data, labels=sorted(df['Year'].unique()))
axes[2].set_title('Yearly Rate Distribution', fontweight='bold')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Rate')

plt.tight_layout()
plt.show()

In [ ]:
# ─── Stationarity Test (ADF Test) ────────────────────────────────────────────
# ARIMA requires the time series to be stationary (constant mean & variance)
# Augmented Dickey-Fuller (ADF) Test:
#   H0: Series is NON-stationary
#   If p-value < 0.05 → reject H0 → series IS stationary

def adf_test(series, name):
    result = adfuller(series.dropna())
    print(f"\n📌 ADF Test for: {name}")
    print(f"   ADF Statistic : {result[0]:.4f}")
    print(f"   p-value       : {result[1]:.4f}")
    if result[1] < 0.05:
        print("   ✅ Series is STATIONARY (p < 0.05)")
    else:
        print("   ⚠️  Series is NON-STATIONARY (p > 0.05) — differencing needed")

adf_test(df['INR_AED'], 'Original INR/AED Rate')

# First-order differencing to make it stationary
df['INR_AED_diff'] = df['INR_AED'].diff()
adf_test(df['INR_AED_diff'], 'Differenced INR/AED Rate')

---
## 3. Preprocessing & Feature Engineering

### Feature Engineering for ML Models (Linear Regression & Random Forest)

Since Linear Regression and Random Forest do not inherently understand time order, we create **lag features** and **rolling statistics** to give the model temporal context:

| Feature | Description |
|---------|-------------|
| `Lag_1` to `Lag_30` | Rate from 1 to 30 days ago |
| `MA_7`, `MA_14`, `MA_30` | 7, 14, 30-day moving averages |
| `Std_7`, `Std_14` | Rolling standard deviations |
| `Day`, `Month`, `DayOfWeek` | Calendar features |

In [ ]:
# ─── Feature Engineering ─────────────────────────────────────────────────────
df_ml = df[['INR_AED']].copy()

# Lag features: yesterday's rate, 2 days ago, etc.
for lag in [1, 2, 3, 5, 7, 14, 21, 30]:
    df_ml[f'Lag_{lag}'] = df_ml['INR_AED'].shift(lag)

# Rolling statistics
df_ml['MA_7']   = df_ml['INR_AED'].shift(1).rolling(7).mean()
df_ml['MA_14']  = df_ml['INR_AED'].shift(1).rolling(14).mean()
df_ml['MA_30']  = df_ml['INR_AED'].shift(1).rolling(30).mean()
df_ml['Std_7']  = df_ml['INR_AED'].shift(1).rolling(7).std()
df_ml['Std_14'] = df_ml['INR_AED'].shift(1).rolling(14).std()

# Calendar features
df_ml['Day']       = df_ml.index.day
df_ml['Month']     = df_ml.index.month
df_ml['DayOfWeek'] = df_ml.index.dayofweek

# Drop NaN rows created by lag/rolling features
df_ml.dropna(inplace=True)

print(f"✅ Feature engineering complete!")
print(f"   Total samples after feature creation: {len(df_ml)}")
print(f"   Total features: {df_ml.shape[1] - 1}")
df_ml.head(3)

In [ ]:
# ─── Train-Test Split ────────────────────────────────────────────────────────
# Use last 20% of data as test set — preserving TIME ORDER is critical!
# We NEVER shuffle time series data.

X = df_ml.drop('INR_AED', axis=1)
y = df_ml['INR_AED']

split = int(len(df_ml) * 0.80)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

test_dates = df_ml.index[split:]

print(f"✅ Train size : {len(X_train)} samples ({X_train.index[0].date()} → {X_train.index[-1].date()})")
print(f"✅ Test  size : {len(X_test)} samples ({X_test.index[0].date()} → {X_test.index[-1].date()})")

In [ ]:
# ─── Helper: Evaluation Metrics ──────────────────────────────────────────────
model_results = {}   # Store all results for final comparison

def evaluate_regression(name, y_true, y_pred, dates=None):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    model_results[name] = {'MAE': mae, 'RMSE': rmse, 'R²': r2, 'MAPE (%)': mape}

    print(f"\n{'='*50}")
    print(f"  {name} — Performance Metrics")
    print(f"{'='*50}")
    print(f"  MAE     : {mae:.6f}  (Mean Absolute Error)")
    print(f"  RMSE    : {rmse:.6f}  (Root Mean Squared Error)")
    print(f"  R²      : {r2:.4f}    (R-Squared Score)")
    print(f"  MAPE    : {mape:.4f}%  (Mean Absolute % Error)")

    # Plot actual vs predicted
    plt.figure(figsize=(13, 4))
    idx = dates if dates is not None else range(len(y_true))
    plt.plot(idx, np.array(y_true), label='Actual',    color='#2980b9', linewidth=1.5)
    plt.plot(idx, np.array(y_pred), label='Predicted', color='#e74c3c', linewidth=1.5, linestyle='--')
    plt.title(f'{name} — Actual vs Predicted INR/AED', fontsize=13, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('INR/AED Rate')
    plt.legend()
    plt.tight_layout()
    plt.show()

print("✅ Evaluation helper ready.")

---
## 4. Linear Regression

### How It Works
Linear Regression finds the best-fit line through the data by minimizing the **sum of squared errors** between actual and predicted values:

$$\hat{y} = w_1x_1 + w_2x_2 + \ldots + w_nx_n + b$$

For time series, we feed **lag features** as inputs so the model learns: *"given the rate over the past N days, predict tomorrow's rate."*

### Strengths & Limitations for Forex
| ✅ Strengths | ⚠️ Limitations |
|-------------|----------------|
| Fast and interpretable | Assumes linear relationships |
| Good baseline model | Cannot capture complex non-linear patterns |
| Works well with lag features | Sensitive to outliers |

In [ ]:
# ─── 4. Linear Regression ────────────────────────────────────────────────────
# fit_intercept=True → learns bias term b
# No scaling needed for LR in this case (tree-based not required either)

lr = LinearRegression(fit_intercept=True)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

evaluate_regression('Linear Regression', y_test, y_pred_lr, test_dates)

In [ ]:
# ─── Top Feature Coefficients ────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': np.abs(lr.coef_)
}).sort_values('Coefficient', ascending=False).head(10)

plt.figure(figsize=(9, 4))
plt.barh(coef_df['Feature'], coef_df['Coefficient'],
         color='#3498db', edgecolor='white')
plt.title('Top 10 Feature Importances — Linear Regression', fontweight='bold')
plt.xlabel('|Coefficient|')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 5. Random Forest Regressor

### How It Works
Random Forest builds an **ensemble of decision trees**, each trained on a random bootstrap sample of the data. For regression, the final prediction is the **average of all trees' predictions**:

$$\hat{y} = \frac{1}{T} \sum_{t=1}^{T} h_t(x)$$

It captures **non-linear relationships** between lag features and future rates — e.g., it can learn *"when the rate has been falling for 7 days AND volatility is high, it tends to rebound."*

### Strengths & Limitations for Forex
| ✅ Strengths | ⚠️ Limitations |
|-------------|----------------|
| Captures non-linear patterns | Cannot extrapolate beyond training range |
| Robust to noise | Slower than Linear Regression |
| Built-in feature importance | May lag behind sudden trend changes |

In [ ]:
# ─── 5. Random Forest Regressor ──────────────────────────────────────────────
# n_estimators=200  → 200 trees (more trees = more stable predictions)
# max_depth=10      → limit depth to prevent overfitting
# min_samples_leaf=5→ each leaf must have at least 5 samples
# random_state=42   → reproducibility

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

evaluate_regression('Random Forest', y_test, y_pred_rf, test_dates)

In [ ]:
# ─── Feature Importances — Random Forest ─────────────────────────────────────
rf_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(9, 4))
plt.barh(rf_imp['Feature'], rf_imp['Importance'],
         color='#27ae60', edgecolor='white')
plt.title('Top 10 Feature Importances — Random Forest', fontweight='bold')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 6. ARIMA (Time Series)

### How It Works
**ARIMA (AutoRegressive Integrated Moving Average)** is a classical statistical model specifically designed for time series forecasting. It has three components:

| Component | Symbol | Description |
|-----------|--------|-------------|
| **AutoRegressive** | p | Uses past p values to predict the next value |
| **Integrated** | d | Number of differencing steps to make series stationary |
| **Moving Average** | q | Uses past q forecast errors to improve prediction |

The model is written as **ARIMA(p, d, q)**. We determine the best `p` and `q` values from **ACF** and **PACF** plots.

### Why Suitable for Forex?
- Exchange rates are classic **time series** data
- ARIMA explicitly models **autocorrelation** (today's rate depends on yesterday's)
- The `d=1` differencing handles the non-stationarity we detected in the ADF test

In [ ]:
# ─── ACF and PACF Plots to determine p and q ─────────────────────────────────
# ACF  → helps determine q (MA order)
# PACF → helps determine p (AR order)

series = df['INR_AED'].dropna()
diff_series = series.diff().dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(series,       lags=40, ax=axes[0][0], title='ACF — Original Series')
plot_pacf(series,      lags=40, ax=axes[0][1], title='PACF — Original Series')
plot_acf(diff_series,  lags=40, ax=axes[1][0], title='ACF — Differenced Series')
plot_pacf(diff_series, lags=40, ax=axes[1][1], title='PACF — Differenced Series')

plt.tight_layout()
plt.show()

print("📌 From ACF/PACF plots: using ARIMA(5, 1, 0) as a solid starting configuration")

In [ ]:
# ─── 6. ARIMA Model ───────────────────────────────────────────────────────────
# We use 80% for training, predict on the remaining 20% using walk-forward validation
# Walk-forward: retrain model each step on all available data up to that point

arima_series = df['INR_AED'].dropna()
arima_split  = int(len(arima_series) * 0.80)

train_arima = arima_series.iloc[:arima_split]
test_arima  = arima_series.iloc[arima_split:]
arima_dates = arima_series.index[arima_split:]

print(f"ARIMA Training: {len(train_arima)} points")
print(f"ARIMA Testing : {len(test_arima)} points")
print("\n⏳ Running ARIMA walk-forward prediction (this may take a minute)...")

history      = list(train_arima)
arima_preds  = []

# Walk-forward validation: predict one step at a time
for i in range(len(test_arima)):
    model = ARIMA(history, order=(5, 1, 0))
    model_fit = model.fit()
    yhat = model_fit.forecast(steps=1)[0]
    arima_preds.append(yhat)
    history.append(test_arima.iloc[i])   # Add actual value to history

    if i % 50 == 0:
        print(f"   Step {i}/{len(test_arima)} completed...")

print("\n✅ ARIMA predictions complete!")
evaluate_regression('ARIMA(5,1,0)', test_arima, arima_preds, arima_dates)

---
## 7. LSTM (Deep Learning)

### How It Works
**LSTM (Long Short-Term Memory)** is a special type of **Recurrent Neural Network (RNN)** designed to learn **long-range dependencies** in sequential data. Regular RNNs suffer from the **vanishing gradient problem**, but LSTMs solve this with a sophisticated **gating mechanism**:

| Gate | Role |
|------|------|
| **Forget Gate** | Decides what past information to discard |
| **Input Gate** | Decides what new information to store |
| **Output Gate** | Decides what to output based on cell state |

The LSTM maintains a **cell state** (long-term memory) and **hidden state** (short-term memory) across time steps.

### Architecture Used
```
Input (60 timesteps) → LSTM(128) → Dropout(0.2) → LSTM(64) → Dropout(0.2) → Dense(32) → Dense(1)
```

### Why Suitable for Forex?
- Exchange rates have **long-range temporal dependencies** (months-long trends)
- LSTM can learn **complex non-linear sequential patterns** automatically
- Outperforms classical models on volatile, noisy financial time series

In [ ]:
# ─── Prepare Data for LSTM ────────────────────────────────────────────────────
# LSTM requires data in shape: (samples, timesteps, features)
# We use a sliding window of 60 days to predict the next day

WINDOW_SIZE = 60   # Use past 60 days to predict next day

# Scale data to [0, 1] — essential for neural networks
lstm_data = df['INR_AED'].values.reshape(-1, 1)
scaler_lstm = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler_lstm.fit_transform(lstm_data)

# Create sliding window sequences
def create_sequences(data, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i, 0])   # Past 60 values
        y.append(data[i, 0])             # Next value
    return np.array(X), np.array(y)

X_seq, y_seq = create_sequences(scaled_data, WINDOW_SIZE)

# Reshape to (samples, timesteps, features=1)
X_seq = X_seq.reshape(X_seq.shape[0], X_seq.shape[1], 1)

# Train-test split (preserving time order)
split_lstm = int(len(X_seq) * 0.80)
X_train_l, X_test_l = X_seq[:split_lstm], X_seq[split_lstm:]
y_train_l, y_test_l = y_seq[:split_lstm], y_seq[split_lstm:]

lstm_test_dates = df.index[WINDOW_SIZE + split_lstm:]

print(f"✅ LSTM data prepared!")
print(f"   Training sequences : {X_train_l.shape}")
print(f"   Testing  sequences : {X_test_l.shape}")

In [ ]:
# ─── Build LSTM Model ─────────────────────────────────────────────────────────
tf.random.set_seed(42)

model_lstm = Sequential([
    # First LSTM layer: 128 units, return_sequences=True to stack another LSTM
    LSTM(128, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
    Dropout(0.2),           # Drop 20% neurons to prevent overfitting

    # Second LSTM layer: 64 units
    LSTM(64, return_sequences=False),
    Dropout(0.2),

    # Fully connected layers
    Dense(32, activation='relu'),
    Dense(1)                # Output: single value (next day's rate)
])

# Compile with Adam optimizer and MSE loss
model_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error'
)

model_lstm.summary()

In [ ]:
# ─── Train LSTM ───────────────────────────────────────────────────────────────
# EarlyStopping: stop training if validation loss doesn't improve for 10 epochs
# This prevents overfitting and saves training time

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history_lstm = model_lstm.fit(
    X_train_l, y_train_l,
    epochs=100,
    batch_size=32,
    validation_split=0.1,   # 10% of training data for validation
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# ─── Training History Plot ────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(history_lstm.history['loss'],     label='Training Loss',   color='#3498db')
plt.plot(history_lstm.history['val_loss'], label='Validation Loss', color='#e74c3c')
plt.title('LSTM Training & Validation Loss', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─── LSTM Predictions ─────────────────────────────────────────────────────────
y_pred_lstm_scaled = model_lstm.predict(X_test_l)

# Inverse transform — convert back from [0,1] scale to actual AED values
y_pred_lstm = scaler_lstm.inverse_transform(y_pred_lstm_scaled).flatten()
y_test_lstm = scaler_lstm.inverse_transform(y_test_l.reshape(-1, 1)).flatten()

evaluate_regression('LSTM', y_test_lstm, y_pred_lstm, lstm_test_dates[:len(y_pred_lstm)])

---
## 8. Model Comparison & Future Prediction

### 8.1 Performance Comparison

We compare all four models using:
- **MAE** — average absolute error in AED units
- **RMSE** — penalizes large errors more
- **R²** — how much variance the model explains (1.0 = perfect)
- **MAPE** — percentage error (easy to interpret)

In [ ]:
# ─── Comparison Table ────────────────────────────────────────────────────────
comp_df = pd.DataFrame(model_results).T.round(6)
comp_df = comp_df.sort_values('RMSE')

print("\n📊 Model Performance Comparison:")
print(comp_df.to_string())

best  = comp_df['RMSE'].idxmin()
worst = comp_df['RMSE'].idxmax()
print(f"\n🏆 Best  Model : {best}  (Lowest RMSE)")
print(f"⚠️  Worst Model : {worst} (Highest RMSE)")

In [ ]:
# ─── Comparison Bar Charts ───────────────────────────────────────────────────
metrics  = ['MAE', 'RMSE', 'MAPE (%)']
colors   = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
models_l = list(comp_df.index)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, metric in enumerate(metrics):
    vals = comp_df[metric]
    bars = axes[i].bar(models_l, vals, color=colors, edgecolor='white', width=0.5)
    axes[i].set_title(metric, fontsize=13, fontweight='bold')
    axes[i].set_ylabel(metric)
    for bar in bars:
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() * 1.01,
                     f"{bar.get_height():.5f}",
                     ha='center', fontsize=8.5)
    plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=15)

plt.suptitle('Model Comparison — Error Metrics (Lower is Better)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─── R² Score Comparison ─────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
r2_vals = comp_df['R²']
bars = plt.bar(models_l, r2_vals, color=colors, edgecolor='white', width=0.5)
plt.axhline(1.0, color='black', linestyle='--', linewidth=0.8, label='Perfect R²=1')
plt.title('R² Score Comparison (Higher is Better)', fontsize=13, fontweight='bold')
plt.ylabel('R² Score')
plt.ylim(0.8, 1.02)
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.002,
             f"{bar.get_height():.4f}",
             ha='center', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─── Future Prediction using LSTM (Next 30 Days) ──────────────────────────────
# Use the best deep learning model to forecast the next 30 days

FORECAST_DAYS = 30

# Start from the last 60 days of real data
last_sequence = scaled_data[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
future_preds  = []

for _ in range(FORECAST_DAYS):
    next_pred = model_lstm.predict(last_sequence, verbose=0)[0][0]
    future_preds.append(next_pred)
    # Roll the window: drop oldest, add new prediction
    last_sequence = np.roll(last_sequence, -1, axis=1)
    last_sequence[0, -1, 0] = next_pred

# Inverse transform to real AED values
future_preds_real = scaler_lstm.inverse_transform(
    np.array(future_preds).reshape(-1, 1)
).flatten()

# Generate future dates
last_date    = df.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=FORECAST_DAYS, freq='B')  # Business days

# Plot historical + forecast
plt.figure(figsize=(14, 5))
plt.plot(df.index[-120:], df['INR_AED'].iloc[-120:],
         color='#2980b9', linewidth=1.5, label='Historical Rate')
plt.plot(future_dates, future_preds_real,
         color='#e74c3c', linewidth=2, linestyle='--',
         marker='o', markersize=4, label=f'LSTM Forecast (Next {FORECAST_DAYS} days)')
plt.axvline(last_date, color='gray', linestyle=':', linewidth=1.5, label='Today')
plt.fill_between(future_dates,
                 future_preds_real * 0.995,
                 future_preds_real * 1.005,
                 color='#e74c3c', alpha=0.15, label='Confidence Band')
plt.title(f'INR/AED — LSTM Forecast for Next {FORECAST_DAYS} Days',
          fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Rate (1 INR = ? AED)')
plt.legend()
plt.tight_layout()
plt.show()

# Print forecast table
forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Predicted INR/AED': future_preds_real.round(6)
})
print("\n📅 30-Day INR/AED Forecast:")
print(forecast_df.to_string(index=False))

---
## 9. Conclusion

### Algorithm Summary

| Model | Strengths | Best For |
|-------|-----------|----------|
| **Linear Regression** | Fast, interpretable, good baseline | Stable trend periods |
| **Random Forest** | Handles non-linearity, robust to noise | Complex pattern capture |
| **ARIMA** | Statistically rigorous, no feature engineering needed | Short-term forecasting |
| **LSTM** | Learns long-range temporal dependencies | Long-term trend prediction |

### Key Takeaways
1. **LSTM** generally achieves the best accuracy for forex prediction due to its ability to learn sequential patterns over long horizons
2. **ARIMA** is the most interpretable and statistically grounded — great for short-term one-step forecasts
3. **Random Forest** with lag features is a strong, fast alternative to deep learning
4. **Feature scaling** (MinMaxScaler) is critical for LSTM convergence
5. **Never shuffle** time series data — always split chronologically
6. Exchange rate prediction is inherently uncertain — these models capture trends, not sudden geopolitical events

### ⚠️ Disclaimer
> This project is for **educational purposes only**. Exchange rate predictions should **NOT** be used as financial advice. Forex markets are influenced by many unpredictable factors beyond historical data.